<a href="https://colab.research.google.com/github/Cshoga/Intro-to-ML_Summative/blob/main/student_mode_12_experiments_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Student Academic Outcome Prediction — ML vs Deep Learning

**Student-mode notebook**

This notebook uses a student academic dataset to predict whether a student is likely to **Dropout**, remain **Enrolled**, or **Graduate**.

The problem is meaningful because education systems can use early prediction to identify students who may need academic, financial, or counselling support.

## What this notebook includes

- Simple data loading and cleaning
- Exploratory Data Analysis
- Preprocessing pipeline
- Traditional Machine Learning experiments using **Scikit-learn**
- Deep Learning experiments using **TensorFlow**
- Both **Sequential API** and **Functional API**
- Use of **tf.data API**
- 12 simple experiments
- Accuracy, precision, recall, F1-score
- Confusion matrices
- ROC curves
- Learning curves
- Final comparison table
- Student-friendly interpretation notes

## 1. Import libraries

This cell imports the tools needed for data analysis, machine learning, deep learning, and visualization.

In [ ]:
# Basic libraries
import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-learn
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay,
    roc_curve, auc
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# TensorFlow
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print("Libraries imported successfully.")
print("TensorFlow version:", tf.__version__)

## 2. Load the dataset

Place `data.csv` in the same folder as this notebook in GitHub/Colab.

The dataset is separated by semicolons, so we use `sep=';'`.

In [ ]:
DATA_PATH = "data.csv"

# If using Colab and the file is not in the current folder, upload it manually or mount Google Drive.
df = pd.read_csv(DATA_PATH, sep=";")

print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()

## 3. Understand the dataset

This cell checks the columns, missing values, and target classes.

**Student note:** The target column is what the model tries to predict.

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

print("\nTarget distribution:")
print(df["Target"].value_counts())

## 4. Clean column names

Some column names contain extra spaces or tab characters. This makes coding harder, so we clean them.

In [ ]:
df.columns = (
    df.columns
    .str.strip()
    .str.replace("\t", "", regex=False)
    .str.replace(" ", "_")
    .str.replace("/", "_")
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
)

print(df.columns.tolist())
df.head()

## 5. Exploratory Data Analysis

This section gives a simple overview of the target classes and selected numerical features.

**Insight to write in report:** If one class is much larger than another, the model may become biased toward the majority class.

In [ ]:
plt.figure(figsize=(6,4))
df["Target"].value_counts().plot(kind="bar")
plt.title("Distribution of Student Outcomes")
plt.xlabel("Outcome")
plt.ylabel("Number of Students")
plt.xticks(rotation=0)
plt.show()

df.describe().T.head(15)

## 6. Simple feature engineering

We create a few useful features from semester performance.

These features are easy to explain:
- Total approved curricular units
- Average grade across semesters
- Total evaluations
- Total enrolled units

In [ ]:
df_fe = df.copy()

df_fe["Total_approved_units"] = (
    df_fe["Curricular_units_1st_sem_approved"] +
    df_fe["Curricular_units_2nd_sem_approved"]
)

df_fe["Average_semester_grade"] = (
    df_fe["Curricular_units_1st_sem_grade"] +
    df_fe["Curricular_units_2nd_sem_grade"]
) / 2

df_fe["Total_evaluations"] = (
    df_fe["Curricular_units_1st_sem_evaluations"] +
    df_fe["Curricular_units_2nd_sem_evaluations"]
)

df_fe["Total_enrolled_units"] = (
    df_fe["Curricular_units_1st_sem_enrolled"] +
    df_fe["Curricular_units_2nd_sem_enrolled"]
)

print("New shape after feature engineering:", df_fe.shape)
df_fe[["Total_approved_units", "Average_semester_grade", "Total_evaluations", "Total_enrolled_units", "Target"]].head()

## 7. Prepare features and target

The target values are converted into numbers because models need numerical labels.

Mapping:
- Dropout = 0
- Enrolled = 1
- Graduate = 2

In [ ]:
target_map = {"Dropout": 0, "Enrolled": 1, "Graduate": 2}
label_map = {0: "Dropout", 1: "Enrolled", 2: "Graduate"}

df_fe["Target_encoded"] = df_fe["Target"].map(target_map)

X = df_fe.drop(columns=["Target", "Target_encoded"])
y = df_fe["Target_encoded"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print(y.value_counts().rename(index=label_map))

## 8. Train-validation-test split

We split the data into:
- Training set: used to train models
- Validation set: used to tune deep learning models
- Test set: used for final evaluation

Stratification keeps the class distribution similar in each split.

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.20, random_state=SEED, stratify=y_train_full
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

## 9. Scale the features

Scaling is important because many models and neural networks perform better when features have similar ranges.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

num_classes = len(np.unique(y))

print("Scaling completed.")
print("Number of classes:", num_classes)

## 10. Helper functions for evaluation

These functions avoid repeating code.

They calculate metrics and produce plots for each experiment.

In [ ]:
results = []

def evaluate_model(name, model_type, y_true, y_pred, y_proba=None):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    results.append({
        "Experiment": name,
        "Type": model_type,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    })

    print(f"===== {name} =====")
    print("Accuracy:", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))
    print("F1-score:", round(f1, 4))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=[label_map[i] for i in range(num_classes)], zero_division=0))

    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred,
        display_labels=[label_map[i] for i in range(num_classes)],
        cmap=None,
        xticks_rotation=45
    )
    plt.title(f"Confusion Matrix - {name}")
    plt.show()

def plot_multiclass_roc(y_true, y_proba, title):
    y_bin = label_binarize(y_true, classes=list(range(num_classes)))

    plt.figure(figsize=(7,5))
    for i in range(num_classes):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{label_map[i]} AUC = {roc_auc:.2f}")

    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.title(title)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.show()

def plot_sklearn_learning_curve(model, X_data, y_data, title):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_data, y_data,
        cv=3,
        scoring="f1_weighted",
        train_sizes=np.linspace(0.2, 1.0, 5),
        n_jobs=-1
    )

    plt.figure(figsize=(7,5))
    plt.plot(train_sizes, train_scores.mean(axis=1), marker="o", label="Training score")
    plt.plot(train_sizes, val_scores.mean(axis=1), marker="o", label="Validation score")
    plt.title(title)
    plt.xlabel("Training examples")
    plt.ylabel("Weighted F1-score")
    plt.legend()
    plt.show()

# Traditional Machine Learning Experiments

We now run 6 simple Scikit-learn experiments.

The goal is not only to get the highest score, but also to compare how different model types behave.

## Experiment 1: Logistic Regression

**Purpose:** Logistic Regression is a simple baseline model. It helps us understand whether the relationship between features and student outcome can be captured with a simpler linear method.

In [ ]:
ml1 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=SEED))
])

ml1.fit(X_train_full, y_train_full)
pred = ml1.predict(X_test)
proba = ml1.predict_proba(X_test)

evaluate_model("E1 Logistic Regression", "Machine Learning", y_test, pred, proba)
plot_multiclass_roc(y_test, proba, "ROC Curve - E1 Logistic Regression")
plot_sklearn_learning_curve(ml1, X_train_full, y_train_full, "Learning Curve - E1 Logistic Regression")

**Student interpretation guide:** After running this cell, compare the training and validation curve. If the training score is high but validation is lower, the model may be overfitting. If both are low, the model may be underfitting.

## Experiment 2: Decision Tree

**Purpose:** Decision Tree is easy to explain because it uses rule-like decisions. A shallow tree can reduce overfitting.

In [ ]:
ml2 = DecisionTreeClassifier(max_depth=5, random_state=SEED)

ml2.fit(X_train_full, y_train_full)
pred = ml2.predict(X_test)
proba = ml2.predict_proba(X_test)

evaluate_model("E2 Decision Tree", "Machine Learning", y_test, pred, proba)
plot_multiclass_roc(y_test, proba, "ROC Curve - E2 Decision Tree")
plot_sklearn_learning_curve(ml2, X_train_full, y_train_full, "Learning Curve - E2 Decision Tree")

**Student interpretation guide:** After running this cell, compare the training and validation curve. If the training score is high but validation is lower, the model may be overfitting. If both are low, the model may be underfitting.

## Experiment 3: Random Forest

**Purpose:** Random Forest combines many decision trees. It is usually stronger than one tree and can handle non-linear patterns.

In [ ]:
ml3 = RandomForestClassifier(
    n_estimators=150,
    max_depth=8,
    random_state=SEED,
    class_weight="balanced"
)

ml3.fit(X_train_full, y_train_full)
pred = ml3.predict(X_test)
proba = ml3.predict_proba(X_test)

evaluate_model("E3 Random Forest", "Machine Learning", y_test, pred, proba)
plot_multiclass_roc(y_test, proba, "ROC Curve - E3 Random Forest")
plot_sklearn_learning_curve(ml3, X_train_full, y_train_full, "Learning Curve - E3 Random Forest")

**Student interpretation guide:** After running this cell, compare the training and validation curve. If the training score is high but validation is lower, the model may be overfitting. If both are low, the model may be underfitting.

## Experiment 4: Gradient Boosting

**Purpose:** Gradient Boosting builds trees gradually, where each new tree tries to correct previous mistakes.

In [ ]:
ml4 = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.08,
    max_depth=3,
    random_state=SEED
)

ml4.fit(X_train_full, y_train_full)
pred = ml4.predict(X_test)
proba = ml4.predict_proba(X_test)

evaluate_model("E4 Gradient Boosting", "Machine Learning", y_test, pred, proba)
plot_multiclass_roc(y_test, proba, "ROC Curve - E4 Gradient Boosting")
plot_sklearn_learning_curve(ml4, X_train_full, y_train_full, "Learning Curve - E4 Gradient Boosting")

**Student interpretation guide:** After running this cell, compare the training and validation curve. If the training score is high but validation is lower, the model may be overfitting. If both are low, the model may be underfitting.

## Experiment 5: Support Vector Machine

**Purpose:** SVM with RBF kernel can learn non-linear boundaries, but it may be slower than simpler models.

In [ ]:
ml5 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(kernel="rbf", C=1.0, probability=True, random_state=SEED))
])

ml5.fit(X_train_full, y_train_full)
pred = ml5.predict(X_test)
proba = ml5.predict_proba(X_test)

evaluate_model("E5 SVM RBF", "Machine Learning", y_test, pred, proba)
plot_multiclass_roc(y_test, proba, "ROC Curve - E5 SVM RBF")
plot_sklearn_learning_curve(ml5, X_train_full, y_train_full, "Learning Curve - E5 SVM RBF")

**Student interpretation guide:** After running this cell, compare the training and validation curve. If the training score is high but validation is lower, the model may be overfitting. If both are low, the model may be underfitting.

## Experiment 6: K-Nearest Neighbors

**Purpose:** KNN predicts using the nearest examples. It is simple, but can struggle when there are many features.

In [ ]:
ml6 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier(n_neighbors=7))
])

ml6.fit(X_train_full, y_train_full)
pred = ml6.predict(X_test)
proba = ml6.predict_proba(X_test)

evaluate_model("E6 KNN", "Machine Learning", y_test, pred, proba)
plot_multiclass_roc(y_test, proba, "ROC Curve - E6 KNN")
plot_sklearn_learning_curve(ml6, X_train_full, y_train_full, "Learning Curve - E6 KNN")

**Student interpretation guide:** After running this cell, compare the training and validation curve. If the training score is high but validation is lower, the model may be overfitting. If both are low, the model may be underfitting.

# Deep Learning Preparation

For deep learning, we use TensorFlow.

The assignment requires:
- TensorFlow Sequential API
- TensorFlow Functional API
- tf.data API

The next cell creates TensorFlow datasets from the scaled NumPy arrays.

In [ ]:
BATCH_SIZE = 32

train_ds = tf.data.Dataset.from_tensor_slices((X_train_scaled.astype("float32"), y_train.values.astype("int32")))
val_ds = tf.data.Dataset.from_tensor_slices((X_val_scaled.astype("float32"), y_val.values.astype("int32")))
test_ds = tf.data.Dataset.from_tensor_slices((X_test_scaled.astype("float32"), y_test.values.astype("int32")))

train_ds = train_ds.shuffle(buffer_size=len(X_train_scaled), seed=SEED).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

input_dim = X_train_scaled.shape[1]

print("tf.data datasets created successfully.")
print("Input dimension:", input_dim)

## Deep learning helper functions

These functions create models, train them, and plot learning curves.

In [ ]:
def compile_model(model, learning_rate=0.001):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

def plot_dl_history(history, title):
    plt.figure(figsize=(7,5))
    plt.plot(history.history["accuracy"], label="Training accuracy")
    plt.plot(history.history["val_accuracy"], label="Validation accuracy")
    plt.title(title + " - Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

    plt.figure(figsize=(7,5))
    plt.plot(history.history["loss"], label="Training loss")
    plt.plot(history.history["val_loss"], label="Validation loss")
    plt.title(title + " - Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

def evaluate_dl_model(name, model):
    y_proba = model.predict(X_test_scaled.astype("float32"))
    y_pred = np.argmax(y_proba, axis=1)

    evaluate_model(name, "Deep Learning", y_test, y_pred, y_proba)
    plot_multiclass_roc(y_test, y_proba, "ROC Curve - " + name)

## Experiment 7: Sequential API - Small Neural Network

**Purpose:** This is the simplest neural network. It tests whether a small deep learning model can learn useful patterns.

In [ ]:
def build_seq_small():
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation="relu"),
        layers.Dense(num_classes, activation="softmax")
    ])
    return compile_model(model, learning_rate=0.001)

dl1 = build_seq_small()
history1 = dl1.fit(train_ds, validation_data=val_ds, epochs=25, verbose=1)

plot_dl_history(history1, "E7 Sequential Small NN")
evaluate_dl_model("E7 Sequential Small NN", dl1)

**Student interpretation guide:** If validation loss starts increasing while training loss keeps decreasing, the neural network is overfitting. Early stopping and dropout can help.

## Experiment 8: Sequential API - Medium Neural Network with Dropout

**Purpose:** Dropout randomly disables some neurons during training. This can reduce overfitting.

In [ ]:
def build_seq_medium_dropout():
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.30),
        layers.Dense(32, activation="relu"),
        layers.Dense(num_classes, activation="softmax")
    ])
    return compile_model(model, learning_rate=0.001)

dl2 = build_seq_medium_dropout()
history2 = dl2.fit(train_ds, validation_data=val_ds, epochs=30, verbose=1)

plot_dl_history(history2, "E8 Sequential Medium NN with Dropout")
evaluate_dl_model("E8 Sequential Medium NN with Dropout", dl2)

**Student interpretation guide:** If validation loss starts increasing while training loss keeps decreasing, the neural network is overfitting. Early stopping and dropout can help.

## Experiment 9: Sequential API - Lower Learning Rate

**Purpose:** A lower learning rate makes smaller updates. It can train more smoothly, but may learn more slowly.

In [ ]:
def build_seq_low_lr():
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(32, activation="relu"),
        layers.Dense(num_classes, activation="softmax")
    ])
    return compile_model(model, learning_rate=0.0005)

dl3 = build_seq_low_lr()
history3 = dl3.fit(train_ds, validation_data=val_ds, epochs=30, verbose=1)

plot_dl_history(history3, "E9 Sequential Lower Learning Rate")
evaluate_dl_model("E9 Sequential Lower Learning Rate", dl3)

**Student interpretation guide:** If validation loss starts increasing while training loss keeps decreasing, the neural network is overfitting. Early stopping and dropout can help.

## Experiment 10: Functional API - Two Hidden Layers

**Purpose:** The Functional API gives more flexibility than Sequential API. Here it is still simple and easy to explain.

In [ ]:
def build_functional_basic():
    inputs = keras.Input(shape=(input_dim,))
    x = layers.Dense(64, activation="relu")(inputs)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    return compile_model(model, learning_rate=0.001)

dl4 = build_functional_basic()
history4 = dl4.fit(train_ds, validation_data=val_ds, epochs=30, verbose=1)

plot_dl_history(history4, "E10 Functional Basic NN")
evaluate_dl_model("E10 Functional Basic NN", dl4)

**Student interpretation guide:** If validation loss starts increasing while training loss keeps decreasing, the neural network is overfitting. Early stopping and dropout can help.

## Experiment 11: Functional API - Batch Normalization

**Purpose:** Batch normalization can stabilize learning and improve training performance.

In [ ]:
def build_functional_batchnorm():
    inputs = keras.Input(shape=(input_dim,))
    x = layers.Dense(128, activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    return compile_model(model, learning_rate=0.001)

dl5 = build_functional_batchnorm()
history5 = dl5.fit(train_ds, validation_data=val_ds, epochs=35, verbose=1)

plot_dl_history(history5, "E11 Functional NN with BatchNorm")
evaluate_dl_model("E11 Functional NN with BatchNorm", dl5)

**Student interpretation guide:** If validation loss starts increasing while training loss keeps decreasing, the neural network is overfitting. Early stopping and dropout can help.

## Experiment 12: Functional API - Early Stopping

**Purpose:** Early stopping stops training when validation performance no longer improves. This helps prevent overfitting.

In [ ]:
def build_functional_earlystop():
    inputs = keras.Input(shape=(input_dim,))
    x = layers.Dense(128, activation="relu")(inputs)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    return compile_model(model, learning_rate=0.001)

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

dl6 = build_functional_earlystop()
history6 = dl6.fit(
    train_ds,
    validation_data=val_ds,
    epochs=60,
    callbacks=[early_stop],
    verbose=1
)

plot_dl_history(history6, "E12 Functional NN with Early Stopping")
evaluate_dl_model("E12 Functional NN with Early Stopping", dl6)

**Student interpretation guide:** If validation loss starts increasing while training loss keeps decreasing, the neural network is overfitting. Early stopping and dropout can help.

# Final Results Comparison

This cell creates a table comparing all 12 experiments.

Use this table in your written report.

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="F1-score", ascending=False).reset_index(drop=True)
results_df

## Plot final comparison

This chart makes it easier to compare experiments visually.

In [ ]:
plt.figure(figsize=(10,5))
plt.bar(results_df["Experiment"], results_df["F1-score"])
plt.title("Final Model Comparison by Weighted F1-score")
plt.xlabel("Experiment")
plt.ylabel("Weighted F1-score")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()

best_model_row = results_df.iloc[0]
print("Best experiment:")
print(best_model_row)

# Student-style discussion notes

Use this section to write your own interpretation after running the notebook.

Example style:

The best-performing model was selected based on weighted F1-score because the dataset has three classes and the classes are not perfectly balanced. Accuracy alone may not be enough because a model can perform well on the majority class while performing poorly on the minority class.

The confusion matrices helped identify which classes were most often confused. In this problem, confusing Dropout and Enrolled students is important because students who are still enrolled may also need support before they drop out.

The traditional machine learning models were useful because they trained quickly and were easier to compare. The deep learning models were useful because they allowed more flexible architectures, but they required more tuning and careful monitoring of validation loss.

A limitation of this dataset is that it may not fully represent all students or all education systems. The dataset contains structured academic and demographic variables, but it may not include personal, emotional, family, or institutional factors that also affect student success.

# Suggested written report structure

Use this as a guide for your 3,500–5,000 word report.

## Introduction
Explain the problem of student dropout and academic success prediction. Explain why early identification of at-risk students is important.

## Literature Review
Discuss previous work on student dropout prediction, educational data mining, machine learning, and deep learning for education.

## Methodology
Describe the dataset, preprocessing, feature engineering, train-validation-test split, ML models, DL models, metrics, and experiment design.

## Results
Present the final comparison table, learning curves, confusion matrices, and ROC curves.

## Discussion
Compare traditional ML and DL models. Explain overfitting, underfitting, class imbalance, and errors.

## Conclusion
Summarize the best model, main findings, limitations, and possible future improvements.

## References
Use IEEE style references.

## Links
GitHub repository: paste your GitHub link here.

Presentation video: paste your video link here.

# Short video presentation guide

Use this simple 5–10 minute flow:

1. Introduce the problem: predicting student outcomes.
2. Explain why it matters to education.
3. Show the dataset and target classes.
4. Explain preprocessing and feature engineering.
5. Present the 6 machine learning experiments.
6. Present the 6 deep learning experiments.
7. Show the final comparison table.
8. Explain the best model.
9. Discuss limitations.
10. End with what you learned.